<a href="https://colab.research.google.com/github/mehakgupta0406/credit-card-customer-churn/blob/main/02_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model preparation

Run notebook 01 first to generate `bank_churn_prepared.csv`, then run these cells from top to bottom. This notebook currently prepares inputs, a stratified split, and categorical encoding. It does not train a model or report predictive performance.

In [1]:
from pathlib import Path
import pandas as pd

prepared_path = Path("bank_churn_prepared.csv")
if not prepared_path.is_file():
    raise FileNotFoundError(
        "Run 01_data_analysis.ipynb through its final export cell first, "
        "then place bank_churn_prepared.csv in this working directory."
    )

df = pd.read_csv(prepared_path)
df.shape

(10127, 22)

In [2]:
X = df.drop(columns=["CLIENTNUM", "Attrition_Flag", "Churn"])
y = df["Churn"]

print("Input data:", X.shape)
print("Target:", y.shape)

Input data: (10127, 19)
Target: (10127,)


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training customers:", X_train.shape[0])
print("Test customers:", X_test.shape[0])

Training customers: 8101
Test customers: 2026


In [4]:
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()

categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numerical columns:", numeric_columns)
print("\nCategorical columns:", categorical_columns)

Numerical columns: ['Customer_Age', 'Dependent_count', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Avg_Open_To_Buy', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio']

Categorical columns: ['Gender', 'Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[
        ("encoding",
         OneHotEncoder(handle_unknown="ignore"),
         categorical_columns)
    ],
    remainder="passthrough"
)

X_train_transformed = ct.fit_transform(X_train)
X_test_transformed = ct.transform(X_test)

print("Training shape:", X_train_transformed.shape)
print("Test shape:", X_test_transformed.shape)

Training shape: (8101, 37)
Test shape: (2026, 37)
